## Building the factor universe and the scoring pipeline

Per the README's build order, this notebook builds and validates every raw factor (the five
foundational ones first, then the rest of the JKP taxonomy and the novel alpha candidates as
they're built) and the scoring pipeline that operates on top of whatever factors exist
(`scoring/zscore.py`, `scoring/combine.py`, `scoring/neutralize.py`), before any of it gets
promoted into `src/`.

Part 3 (the factors) grows over time as new ones are built; Part 4 (scoring) stays last,
since it's generic machinery that consumes the factor set rather than something tied to a
fixed number of them.

Each part follows the same shape: build a small piece, validate it against a toy case with a
known answer, then check it against a real sample before it's trusted enough to promote. Real
bugs found along the way (a ticker misattribution in `ticker_on`, a stock-split gap in market
cap) are fixed at the source and documented in `notebooks/logs/` and the relevant module's own
docstring, not replayed here; this notebook stays focused on validating what exists today.

## Part 1: setup

Universe, fundamentals, and price caches, plus a fixed, reproducible sample of real companies
reused throughout the rest of this notebook for real-data checks.

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import math
import random

import pandas as pd

from src.universe.point_in_time import build_universe, ticker_on
from src.loaders.fundamentals import build_fundamentals, load_company_facts, latest_value_as_of
from src.loaders.prices import build_prices, load_cik_prices, close_on_or_before
from src.factors.size import market_cap_as_of, split_adjustment_ratio, size_factor
from src.factors.value import earnings_yield_factor
from src.factors.quality import roe_factor
from src.factors.momentum import momentum_factor
from src.factors.low_vol import low_vol_factor
from src.scoring.zscore import winsorize, zscore
from src.scoring.combine import combine
from src.scoring.neutralize import neutralize

universe_spans, ticker_history = build_universe()
build_fundamentals()
build_prices()

AS_OF = "2024-06-28"   # a recent Friday, arbitrary
random.seed(0)          # fixed, so real-data checks below are reproducible run to run

active = universe_spans[
    (universe_spans["start_date"] <= AS_OF)
    & (universe_spans["end_date"].isna() | (universe_spans["end_date"] >= AS_OF))
]
sample_ciks = random.sample(list(active["cik"].dropna().unique()), 60)

## Part 2: price and market cap helpers

`close_on_or_before` (point in time close, never a later date), `split_adjustment_ratio`, and
`market_cap_as_of` now live in `src/loaders/prices.py` and `src/factors/size.py`. The real bug
that motivated `split_adjustment_ratio`: a filed share count and a cached price come from two
different sources with two different split bases, and combining them directly silently
understated market cap by the cumulative split ratio for any company that split its stock
between the filing and today. Confirmed on CMG (a real 50-for-1 split) and ORLY, whose split
happened chronologically *after* `AS_OF` and still corrupted the result, since cached prices
are always adjusted to whenever they were fetched, not to `AS_OF`.

In [2]:
toy_prices = pd.DataFrame({
    "ticker": ["XYZ", "XYZ", "XYZ"],
    "Close": [10.0, 11.0, 12.0],
}, index=pd.to_datetime(["2024-01-05", "2024-01-08", "2024-01-09"]).tz_localize("America/New_York"))
# 2024-01-05 is a Friday, 2024-01-08 a Monday: a real weekend gap in between.

print(close_on_or_before(toy_prices, "XYZ", "2024-01-08"))  # exact day: expect 11.0
print(close_on_or_before(toy_prices, "XYZ", "2024-01-07"))  # a Sunday: expect 10.0, Friday's close
print(close_on_or_before(toy_prices, "XYZ", "2024-01-01"))  # before any data: expect None
print(close_on_or_before(toy_prices, "ABC", "2024-01-08"))  # ticker not present: expect None

cmg_prices = load_cik_prices(1058090)
print(split_adjustment_ratio(cmg_prices, "CMG", "2024-04-22"))    # real 50-for-1 split; expect 50.0

orly_prices = load_cik_prices(898173)
print(split_adjustment_ratio(orly_prices, "ORLY", "2024-04-29"))  # expect 15.0, split after AS_OF

aapl_prices = load_cik_prices(320193)
print(split_adjustment_ratio(aapl_prices, "AAPL", "2024-06-28"))  # expect 1.0: no split since 2020

11.0
10.0
None
None
50.0
15.0
1.0


## Part 3: the factors

Each raw factor returns the plain quantity it's named for (log market cap, an earnings yield,
a return on equity, a trailing return, a standard deviation), never negated or rescaled for its
expected direction of alpha. Which direction to bet is a decision for the alpha model and
optimizer later, not something baked into a raw factor's sign here.

3a through 3e are the five foundational factors; later subparts continue with the rest of the
JKP taxonomy and the novel alpha candidates as they're built, per the README's build order.

### 3a. Size: log market capitalization

In [3]:
print(size_factor(math.e))   # log(e) = 1, exact
print(size_factor(1.0))      # log(1) = 0, exact
print(size_factor(None))     # expect None
print(size_factor(-100.0))   # expect None, defensive

rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if facts is None or ticker is None or prices is None:
        continue
    sf = size_factor(market_cap_as_of(facts, prices, ticker, AS_OF))
    if sf is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "size_factor": sf})

size_df = pd.DataFrame(rows)
print(f"{len(size_df)} / {len(sample_ciks)} resolved")
print(size_df["size_factor"].describe())

1.0
0.0
None
None
60 / 60 resolved
count    60.000000
mean     24.423091
std       1.344796
min      22.719638
25%      23.623812
50%      24.131652
75%      24.843289
max      30.379024
Name: size_factor, dtype: float64


### 3b. Value: earnings yield

Net income over split-adjusted market cap. Earnings, not revenue or gross profit, as the
numerator, since `net_income` is tagged consistently across nearly every filer where the other
two have documented sector gaps (banks report interest income instead of revenue).

In [4]:
toy_value_facts = {
    "facts": {
        "dei": {
            "EntityCommonStockSharesOutstanding": {
                "units": {"shares": [{"end": "2024-04-22", "val": 1000.0, "filed": "2024-04-25", "form": "10-Q"}]}
            }
        },
        "us-gaap": {
            "NetIncomeLoss": {
                "units": {"USD": [
                    {"start": "2023-01-01", "end": "2023-12-31", "val": 500.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            }
        },
    }
}
toy_value_prices = pd.DataFrame({
    "ticker": ["XYZ", "XYZ", "XYZ"],
    "Close": [100.0, 2.0, 2.1],
    "Stock Splits": [0.0, 50.0, 0.0],
}, index=pd.to_datetime(["2024-04-22", "2024-06-26", "2024-06-28"]).tz_localize("America/New_York"))

result = earnings_yield_factor(toy_value_facts, toy_value_prices, "XYZ", "2024-06-28")
print(result)   # expect 500.0 / (1000.0 * 50.0 * 2.1) = 500.0 / 105000.0 ≈ 0.004762

rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if facts is None or ticker is None or prices is None:
        continue
    ey = earnings_yield_factor(facts, prices, ticker, AS_OF)
    if ey is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "earnings_yield": ey})

value_df = pd.DataFrame(rows)
print(f"{len(value_df)} / {len(sample_ciks)} resolved")
print(value_df["earnings_yield"].describe())

0.004761904761904762
60 / 60 resolved
count    60.000000
mean      0.054543
std       0.057622
min      -0.167039
25%       0.024927
50%       0.041915
75%       0.092214
max       0.216984
Name: earnings_yield, dtype: float64


### 3c. Quality: return on equity

Negative equity (leveraged buybacks) is deliberately not filtered out: it's real, common data,
not an error, left for `scoring/zscore.py`'s winsorization and later IC measurement to handle.

In [5]:
toy_quality_facts = {
    "facts": {
        "us-gaap": {
            "NetIncomeLoss": {
                "units": {"USD": [
                    {"start": "2023-01-01", "end": "2023-12-31", "val": 500.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
            "StockholdersEquity": {
                "units": {"USD": [
                    {"end": "2023-12-31", "val": 2500.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
        }
    }
}
print(roe_factor(toy_quality_facts, "2024-06-28"))   # expect 500.0 / 2500.0 = 0.2

rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    if facts is None:
        continue
    ticker = ticker_on(ticker_history, cik, AS_OF)
    roe = roe_factor(facts, AS_OF)
    if roe is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "roe": roe})

quality_df = pd.DataFrame(rows)
print(f"{len(quality_df)} / {len(sample_ciks)} resolved")
print(quality_df["roe"].describe())

0.2
60 / 60 resolved
count    60.000000
mean      0.182340
std       0.568810
min      -2.908012
25%       0.098305
50%       0.174745
75%       0.318841
max       1.611324
Name: roe, dtype: float64


### 3d. Momentum: trailing 12 month return, skipping the most recent month

The most recent month is skipped deliberately: short term reversal works in the opposite
direction to momentum over roughly a one month horizon, so including it would blend two
factors with opposite signs into one noisy signal.

In [6]:
toy_momentum_prices = pd.DataFrame({
    "ticker": ["XYZ", "XYZ"],
    "Close": [100.0, 150.0],
}, index=pd.to_datetime(["2023-06-28", "2024-05-28"]).tz_localize("America/New_York"))
print(momentum_factor(toy_momentum_prices, "XYZ", "2024-06-28"))   # expect 150/100 - 1 = 0.5

rows = []
for cik in sample_ciks:
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if ticker is None or prices is None:
        continue
    mom = momentum_factor(prices, ticker, AS_OF)
    if mom is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "momentum": mom})

momentum_df = pd.DataFrame(rows)
print(f"{len(momentum_df)} / {len(sample_ciks)} resolved")
print(momentum_df["momentum"].describe())

0.5
58 / 60 resolved
count    58.000000
mean      0.170201
std       0.262205
min      -0.461712
25%       0.003896
50%       0.128419
75%       0.360626
max       0.774349
Name: momentum, dtype: float64


### 3e. Low volatility: standard deviation of trailing daily returns

252 trading days (about a year), matching momentum's own lookback: more observations give a
materially less noisy standard deviation estimate. Returns the raw standard deviation, not its
negative, same raw-quantity convention as every other factor above.

In [7]:
toy_lowvol_prices = pd.DataFrame({
    "ticker": ["XYZ"] * 5,
    "Close": [100.0, 110.0, 99.0, 108.9, 98.01],
}, index=pd.date_range("2024-06-24", periods=5, freq="B").tz_localize("America/New_York"))
result = low_vol_factor(toy_lowvol_prices, "XYZ", "2024-06-28", lookback_days=4)
expected = pd.Series([0.1, -0.1, 0.1, -0.1]).std()
print(result, expected)   # both should match (small floating point noise aside)

rows = []
for cik in sample_ciks:
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if ticker is None or prices is None:
        continue
    vol = low_vol_factor(prices, ticker, AS_OF)
    if vol is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "low_vol": vol})

lowvol_df = pd.DataFrame(rows)
print(f"{len(lowvol_df)} / {len(sample_ciks)} resolved")
print(lowvol_df["low_vol"].describe())

0.11547005383792519 0.11547005383792516
60 / 60 resolved
count    60.000000
mean      0.017085
std       0.005787
min       0.010304
25%       0.012835
50%       0.016456
75%       0.018927
max       0.040233
Name: low_vol, dtype: float64


### 3f. Short term reversal: negative of trailing one week return


In [8]:
def short_term_reversal_factor(prices, ticker, as_of, lookback=pd.DateOffset(weeks=1)):
    """Raw short term reversal factor: negative of the trailing return over
    lookback (one week by default; pass pd.DateOffset(months=1) for the
    monthly variant Jegadeesh 1990 also documents).

    The negation is part of the factor's own definition, not a later
    alpha-direction choice the way size_factor/low_vol_factor's raw sign
    is: short term reversal bets that a stock's most recent return
    predicts the opposite next, so a name that just fell gets a high score
    here, not a low one, by construction.

    No month skipped, unlike momentum: this factor is the near-term effect
    momentum deliberately excludes, not a longer-horizon signal that needs
    protecting from it.

    Well documented (Jegadeesh 1990, Lehmann 1990) but strongest among
    small, illiquid securities and substantially eroded by transaction
    costs; screen by information coefficient within a large-cap universe
    before trusting it here, per the README, rather than assuming it
    carries over.

    Returns None if either endpoint's close price cannot be resolved.
    """
    end = close_on_or_before(prices, ticker, as_of)
    start = close_on_or_before(prices, ticker, pd.Timestamp(as_of) - lookback)
    if end is None or start is None:
        return None
    return -(end / start - 1)


In [9]:
toy_reversal_prices = pd.DataFrame({
    "ticker": ["XYZ", "XYZ"],
    "Close": [100.0, 90.0],
}, index=pd.to_datetime(["2024-06-21", "2024-06-28"]).tz_localize("America/New_York"))
print(short_term_reversal_factor(toy_reversal_prices, "XYZ", "2024-06-28"))   # expect -(90/100 - 1) = 0.10


0.09999999999999998


In [10]:
rows = []
for cik in sample_ciks:
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if ticker is None or prices is None:
        continue
    rev = short_term_reversal_factor(prices, ticker, AS_OF)
    if rev is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "reversal": rev})

reversal_df = pd.DataFrame(rows)
print(f"{len(reversal_df)} / {len(sample_ciks)} resolved")
print(reversal_df["reversal"].describe())
print(reversal_df.sort_values("reversal").head(5)[["ticker", "reversal"]])
print(reversal_df.sort_values("reversal").tail(5)[["ticker", "reversal"]])


60 / 60 resolved
count    60.000000
mean     -0.002461
std       0.037572
min      -0.188521
25%      -0.013058
50%       0.002445
75%       0.014077
max       0.090524
Name: reversal, dtype: float64
   ticker  reversal
0     FDX -0.188521
48   TSLA -0.081252
28    SYF -0.061165
21   PANW -0.058315
29    KEY -0.044853
   ticker  reversal
57   VLTO  0.033498
53    DAL  0.039287
8     COR  0.048082
33   PAYX  0.058749
39   POOL  0.090524


### 3g. Seasonality: average historical return for the current calendar month


In [11]:
def seasonality_factor(prices, ticker, as_of, min_years=3):
    """Raw seasonality factor: average historical return in this ticker's
    own history for the current calendar month, across every complete
    prior occurrence of that month.

    Deliberately excludes the current, in-progress occurrence of the
    target month: only full month-end-to-month-end returns from strictly
    earlier years count, never a partial return from the month currently
    underway, which would look ahead into data not yet known as of as_of.

    Requires at least min_years complete prior occurrences before
    returning a value, since one or two data points make for an
    unreliable average of what's meant to be a genuinely repeating
    pattern, not a single company-specific event mistaken for one.

    Returns None if fewer than min_years complete occurrences exist.
    """
    as_of_ts = pd.Timestamp(as_of)

    returns = []
    # 40 years comfortably exceeds the universe's own ~28 year horizon
    # (1996 to present); years with no data simply contribute nothing.
    for years_back in range(1, 41):
        month_end = (as_of_ts - pd.DateOffset(years=years_back)).replace(day=1) + pd.offsets.MonthEnd(0)
        prior_month_end = month_end - pd.offsets.MonthEnd(1)
        end_close = close_on_or_before(prices, ticker, month_end)
        start_close = close_on_or_before(prices, ticker, prior_month_end)
        if end_close is None or start_close is None:
            continue
        returns.append(end_close / start_close - 1)

    if len(returns) < min_years:
        return None
    return sum(returns) / len(returns)


In [12]:
toy_seasonality_prices = pd.DataFrame({
    "ticker": ["XYZ"] * 6,
    "Close": [100.0, 110.0, 100.0, 90.0, 100.0, 120.0],
}, index=pd.to_datetime([
    "2021-05-31", "2021-06-30",
    "2022-05-31", "2022-06-30",
    "2023-05-31", "2023-06-30",
]))
# June returns: 2021 +0.10, 2022 -0.10, 2023 +0.20; average = 0.20 / 3 ≈ 0.0667
print(seasonality_factor(toy_seasonality_prices, "XYZ", "2024-06-28"))


0.0666666666666667


In [13]:
rows = []
for cik in sample_ciks:
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if ticker is None or prices is None:
        continue
    seas = seasonality_factor(prices, ticker, AS_OF)
    if seas is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "seasonality": seas})

seasonality_df = pd.DataFrame(rows)
print(f"{len(seasonality_df)} / {len(sample_ciks)} resolved")
print(seasonality_df["seasonality"].describe())
print(seasonality_df.sort_values("seasonality").head(5)[["ticker", "seasonality"]])
print(seasonality_df.sort_values("seasonality").tail(5)[["ticker", "seasonality"]])


56 / 60 resolved
count    56.000000
mean      0.005436
std       0.027431
min      -0.038220
25%      -0.011390
50%      -0.000834
75%       0.010748
max       0.112864
Name: seasonality, dtype: float64
   ticker  seasonality
5    MKTX    -0.038220
52   NXPI    -0.030410
6     CFG    -0.028374
26    KEY    -0.026709
24   SCHW    -0.019525
   ticker  seasonality
36   POOL     0.038823
7    ORCL     0.057231
55   PAYC     0.069182
45   TSLA     0.086292
4     DPZ     0.112864


### 3h. 52 week high proximity: price over trailing 252 day maximum


In [14]:
def high_proximity_factor(prices, ticker, as_of, lookback_days=252):
    """Raw 52 week high proximity factor: price divided by its own trailing
    lookback_days maximum, inclusive of the current price itself.

    A value of 1.0 means today's close is the trailing high; values below
    1.0 measure how far the current price sits below it. Requires at least
    half of lookback_days worth of trading days actually present, same
    threshold and reasoning as low_vol_factor: too few observations means
    the window doesn't really cover a year.

    Returns None if fewer than half of lookback_days trading days are
    present, or if the ticker has no data at all.
    """
    ticker_prices = prices[prices["ticker"] == ticker].sort_index()
    if ticker_prices.empty:
        return None
    as_of_ts = pd.Timestamp(as_of).tz_localize(ticker_prices.index.tz)
    window = ticker_prices.loc[:as_of_ts].tail(lookback_days)["Close"]
    if len(window) < lookback_days // 2:
        return None
    return window.iloc[-1] / window.max()


In [15]:
toy_high_prices = pd.DataFrame({
    "ticker": ["XYZ"] * 5,
    "Close": [100.0, 120.0, 90.0, 80.0, 96.0],
}, index=pd.date_range("2024-06-24", periods=5, freq="B").tz_localize("America/New_York"))
print(high_proximity_factor(toy_high_prices, "XYZ", "2024-06-28", lookback_days=5))   # max=120, current=96: expect 0.8


0.8


In [16]:
rows = []
for cik in sample_ciks:
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if ticker is None or prices is None:
        continue
    hp = high_proximity_factor(prices, ticker, AS_OF)
    if hp is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "high_proximity": hp})

high_df = pd.DataFrame(rows)
print(f"{len(high_df)} / {len(sample_ciks)} resolved")
print(high_df["high_proximity"].describe())
print(high_df.sort_values("high_proximity").head(5)[["ticker", "high_proximity"]])
print(high_df.sort_values("high_proximity").tail(5)[["ticker", "high_proximity"]])


60 / 60 resolved
count    60.000000
mean      0.894457
std       0.112872
min       0.388741
25%       0.872183
50%       0.933829
75%       0.966542
max       1.000000
Name: high_proximity, dtype: float64
   ticker  high_proximity
59   PAYC        0.388741
1      GL        0.642200
48   TSLA        0.674576
5    MKTX        0.681058
38    BAX        0.699757
   ticker  high_proximity
9    TRGP        0.991913
50    LLY        0.995974
46    IRM        1.000000
28    SYF        1.000000
0     FDX        1.000000


### 3i. Volume shock: recent volume over trailing average volume


In [17]:
def volume_shock_factor(prices, ticker, as_of, lookback_days=20):
    """Raw volume shock factor: the most recent day's volume divided by the
    trailing lookback_days average volume immediately before it.

    The trailing average excludes the most recent day itself, so the
    numerator and denominator describe genuinely distinct periods: "is
    today's volume unusual relative to the recent past," not partly
    compared against itself.

    20 trading days (about a month) by default: this factor lives in the
    README's weekly horizon table, a much shorter baseline than momentum
    or low_vol's 252 day window, since a volume shock is inherently a
    fast-changing signal, not a slow-moving one.

    Requires at least half of lookback_days worth of trailing days
    actually present in the baseline window, same reasoning as
    low_vol_factor/high_proximity_factor. Returns None below that, or if
    the ticker has no data at all, or if the trailing average volume is
    zero (a genuinely halted or untraded name, where the ratio is
    undefined rather than infinite).
    """
    ticker_prices = prices[prices["ticker"] == ticker].sort_index()
    if ticker_prices.empty:
        return None
    as_of_ts = pd.Timestamp(as_of).tz_localize(ticker_prices.index.tz)
    window = ticker_prices.loc[:as_of_ts].tail(lookback_days + 1)["Volume"]
    if len(window) < 2:
        return None
    recent = window.iloc[-1]
    baseline = window.iloc[:-1]
    if len(baseline) < lookback_days // 2:
        return None
    baseline_avg = baseline.mean()
    if baseline_avg == 0:
        return None
    return recent / baseline_avg


In [18]:
toy_volume_prices = pd.DataFrame({
    "ticker": ["XYZ"] * 6,
    "Volume": [100, 100, 100, 100, 100, 300],
}, index=pd.date_range("2024-06-21", periods=6, freq="B").tz_localize("America/New_York"))
print(volume_shock_factor(toy_volume_prices, "XYZ", "2024-06-28", lookback_days=5))   # baseline avg 100, recent 300: expect 3.0


3.0


In [19]:
rows = []
for cik in sample_ciks:
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if ticker is None or prices is None:
        continue
    vs = volume_shock_factor(prices, ticker, AS_OF)
    if vs is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "volume_shock": vs})

volume_shock_df = pd.DataFrame(rows)
print(f"{len(volume_shock_df)} / {len(sample_ciks)} resolved")
print(volume_shock_df["volume_shock"].describe())
print(volume_shock_df.sort_values("volume_shock").head(5)[["ticker", "volume_shock"]])
print(volume_shock_df.sort_values("volume_shock").tail(5)[["ticker", "volume_shock"]])


60 / 60 resolved
count    60.000000
mean      1.838387
std       0.858680
min       0.930141
25%       1.303317
50%       1.622668
75%       2.139242
max       6.123892
Name: volume_shock, dtype: float64
   ticker  volume_shock
59   PAYC      0.930141
17     KR      0.978545
37    SJM      1.043812
2     FIS      1.088797
54   COST      1.094518
   ticker  volume_shock
33   PAYX      3.075527
45    LMT      3.315748
13      A      3.447094
26   PCAR      4.223614
57   VLTO      6.123892


### 3j. Illiquidity (Amihud): mean absolute return over dollar volume


In [20]:
def illiquidity_factor(prices, ticker, as_of, lookback_days=20):
    """Raw illiquidity factor (Amihud 2002): mean of daily absolute return
    divided by dollar volume, over a trailing window.

    Higher values mean a given dollar of trading moves the price more,
    i.e. less liquid. Same 20 trading day window as volume_shock_factor:
    like that factor, this lives in the README's weekly horizon table, a
    fast-changing signal rather than a slow one.

    Requires at least half of lookback_days worth of trailing days
    actually present, same reasoning as every other rolling-window factor
    here. A day with zero dollar volume is dropped from the average
    rather than producing an infinite ratio, since a halted or untraded
    day says nothing about liquidity on days that did trade.

    Returns None if fewer than half of lookback_days days remain after
    dropping zero-volume days, or if the ticker has no data at all.
    """
    ticker_prices = prices[prices["ticker"] == ticker].sort_index()
    if ticker_prices.empty:
        return None
    as_of_ts = pd.Timestamp(as_of).tz_localize(ticker_prices.index.tz)
    window = ticker_prices.loc[:as_of_ts].tail(lookback_days + 1)
    if len(window) < 2:
        return None

    returns = window["Close"].pct_change().dropna()
    dollar_volume = (window["Close"] * window["Volume"]).reindex(returns.index)

    ratios = (returns.abs() / dollar_volume)[dollar_volume > 0]
    if len(ratios) < lookback_days // 2:
        return None
    return ratios.mean()


In [21]:
toy_illiq_prices = pd.DataFrame({
    "ticker": ["XYZ"] * 4,
    "Close": [100.0, 110.0, 99.0, 108.9],
    "Volume": [1000, 1000, 1000, 1000],
}, index=pd.date_range("2024-06-25", periods=4, freq="B").tz_localize("America/New_York"))

result = illiquidity_factor(toy_illiq_prices, "XYZ", "2024-06-28", lookback_days=3)
expected = ((0.10 / 110000) + (0.10 / 99000) + (0.10 / 108900)) / 3
print(result, expected)   # both should match


9.458218549127645e-07 9.458218549127641e-07


In [22]:
rows = []
for cik in sample_ciks:
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if ticker is None or prices is None:
        continue
    illiq = illiquidity_factor(prices, ticker, AS_OF)
    if illiq is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "illiquidity": illiq})

illiquidity_df = pd.DataFrame(rows)
print(f"{len(illiquidity_df)} / {len(sample_ciks)} resolved")
print(illiquidity_df["illiquidity"].describe())
print(illiquidity_df.sort_values("illiquidity").head(5)[["ticker", "illiquidity"]])
print(illiquidity_df.sort_values("illiquidity").tail(5)[["ticker", "illiquidity"]])


60 / 60 resolved
count    6.000000e+01
mean     5.496029e-11
std      3.812685e-11
min      1.237071e-12
25%      2.771117e-11
50%      4.686185e-11
75%      7.843564e-11
max      1.744717e-10
Name: illiquidity, dtype: float64
   ticker   illiquidity
48   TSLA  1.237071e-12
20   AVGO  3.252706e-12
50    LLY  3.388398e-12
54   COST  4.210587e-12
7    ORCL  1.036041e-11
   ticker   illiquidity
16   JKHY  1.168940e-10
28    SYF  1.210207e-10
47    ROL  1.311750e-10
43    TPR  1.472074e-10
5    MKTX  1.744717e-10


### 3k. Profitability: gross profit over total assets


In [23]:
def gross_profitability_factor(facts, as_of):
    """Raw profitability factor: gross profit over total assets (Novy-Marx
    2013's gross profitability premium), distinct from quality.py's
    return on equity: a different construct, profitability relative to
    the asset base a company deploys, not relative to its book equity.

    gross_profit resolves via its own revenue-minus-cost-of-revenue
    fallback for filers that never tag GrossProfit directly (e.g.
    DoorDash), automatically, the same DERIVED_FALLBACK machinery
    latest_value_as_of already applies to total_liabilities.

    Unlike quality.py's stockholders_equity, a non-positive total_assets
    is filtered here rather than preserved: a listed company reporting
    zero or negative total assets isn't a real, common state the way
    negative book equity is, so it's treated as an error rather than
    left for scoring/zscore.py to handle.

    Returns None if gross profit or total assets cannot be resolved, or
    if total assets is non-positive.
    """
    gp = latest_value_as_of(facts, "gross_profit", "USD", as_of, period="annual")
    if gp is None:
        return None
    assets = latest_value_as_of(facts, "total_assets", "USD", as_of)
    if assets is None or assets[0] <= 0:
        return None
    return gp[0] / assets[0]


In [24]:
toy_profitability_facts = {
    "facts": {
        "us-gaap": {
            "GrossProfit": {
                "units": {"USD": [
                    {"start": "2023-01-01", "end": "2023-12-31", "val": 400.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
            "Assets": {
                "units": {"USD": [
                    {"end": "2023-12-31", "val": 2000.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
        }
    }
}
print(gross_profitability_factor(toy_profitability_facts, "2024-06-28"))   # direct tag: expect 400/2000 = 0.2

toy_derived_facts = {
    "facts": {
        "us-gaap": {
            "Revenues": {
                "units": {"USD": [
                    {"start": "2023-01-01", "end": "2023-12-31", "val": 1000.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
            "CostOfRevenue": {
                "units": {"USD": [
                    {"start": "2023-01-01", "end": "2023-12-31", "val": 600.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
            "Assets": {
                "units": {"USD": [
                    {"end": "2023-12-31", "val": 2000.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
        }
    }
}
print(gross_profitability_factor(toy_derived_facts, "2024-06-28"))   # derived: (1000-600)/2000 = 0.2, same answer


0.2
0.2


In [25]:
rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    if facts is None:
        continue
    ticker = ticker_on(ticker_history, cik, AS_OF)
    gprof = gross_profitability_factor(facts, AS_OF)
    if gprof is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "gross_profitability": gprof})

profitability_df = pd.DataFrame(rows)
print(f"{len(profitability_df)} / {len(sample_ciks)} resolved")
print(profitability_df["gross_profitability"].describe())
print(profitability_df.sort_values("gross_profitability").head(5)[["ticker", "gross_profitability"]])
print(profitability_df.sort_values("gross_profitability").tail(5)[["ticker", "gross_profitability"]])


44 / 60 resolved
count    44.000000
mean      0.286838
std       0.188278
min       0.044099
25%       0.154117
50%       0.228250
75%       0.396772
max       0.990070
Name: gross_profitability, dtype: float64
   ticker  gross_profitability
1      GL             0.044099
42    EXR             0.070895
15    NEM             0.092401
2     FIS             0.102478
19   NDAQ             0.128688
   ticker  gross_profitability
23     IT             0.519302
34    ROL             0.603098
0     FDX             0.615916
13     KR             0.646852
3     DPZ             0.990070


### 3l. Profit growth: change in net income over total assets


In [26]:
def profit_growth_factor(facts, as_of):
    """Raw profit growth factor: change in net income between the two most
    recently available annual periods, scaled by total assets.

    Scaled by total assets rather than expressed as a percent change of
    the prior period's own net income, deliberately: net income can be
    zero, negative, or small enough that a percent-change denominator
    blows up or flips sign in a way that says nothing about genuine
    growth, the same near-zero-denominator concern that shaped
    market_cap_as_of and quality.py's design. Total assets is virtually
    always positive and comparatively stable, a well behaved denominator.

    Uses latest_value_as_of's offset argument (offset=0 for the most
    recent annual period, offset=1 for the one before it), the exact
    mechanism the fundamentals loader already built and tested for this
    purpose. Both figures use the same as_of query date but different
    offsets, not two different as_of dates, so the "prior" period is
    whichever one this filer's own available annual periods put there,
    not a fixed calendar year back.

    Returns None if either net income figure or total assets cannot be
    resolved, or if total assets is non-positive.
    """
    current = latest_value_as_of(facts, "net_income", "USD", as_of, period="annual", offset=0)
    prior = latest_value_as_of(facts, "net_income", "USD", as_of, period="annual", offset=1)
    if current is None or prior is None:
        return None
    assets = latest_value_as_of(facts, "total_assets", "USD", as_of)
    if assets is None or assets[0] <= 0:
        return None
    return (current[0] - prior[0]) / assets[0]


In [27]:
toy_growth_facts = {
    "facts": {
        "us-gaap": {
            "NetIncomeLoss": {
                "units": {"USD": [
                    {"start": "2022-01-01", "end": "2022-12-31", "val": 300.0, "filed": "2023-02-01", "form": "10-K"},
                    {"start": "2023-01-01", "end": "2023-12-31", "val": 500.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
            "Assets": {
                "units": {"USD": [
                    {"end": "2023-12-31", "val": 1000.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
        }
    }
}
print(profit_growth_factor(toy_growth_facts, "2024-06-28"))   # (500 - 300) / 1000 = 0.2


0.2


In [28]:
rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    if facts is None:
        continue
    ticker = ticker_on(ticker_history, cik, AS_OF)
    pg = profit_growth_factor(facts, AS_OF)
    if pg is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "profit_growth": pg})

profit_growth_df = pd.DataFrame(rows)
print(f"{len(profit_growth_df)} / {len(sample_ciks)} resolved")
print(profit_growth_df["profit_growth"].describe())
print(profit_growth_df.sort_values("profit_growth").head(5)[["ticker", "profit_growth"]])
print(profit_growth_df.sort_values("profit_growth").tail(5)[["ticker", "profit_growth"]])


60 / 60 resolved
count    60.000000
mean      0.020027
std       0.058228
min      -0.059921
25%      -0.002127
50%       0.003077
75%       0.022294
max       0.280617
Name: profit_growth, dtype: float64
   ticker  profit_growth
39   POOL      -0.059921
24    VLO      -0.043037
19    NEM      -0.037318
49    PKG      -0.029875
50    LLY      -0.015708
   ticker  profit_growth
12     MO       0.064866
38    BAX       0.183137
51   EBAY       0.188440
41    EMR       0.215073
2     FIS       0.280617


### 3m. Investment: percent change in total assets, year over year


In [29]:
def investment_factor(facts, as_of, periods_back=4):
    """Raw investment factor: percent change in total assets between the
    most recent balance sheet and the one roughly a year earlier.

    total_assets is an instant concept reported every quarter, so
    consecutive offsets (offset=0, offset=1, ...) step through
    consecutive quarters, not years; offset=4 is what reaches roughly one
    year back for a filer that reports all four quarters separately, the
    same reasoning latest_value_as_of's own docstring gives for its
    offset argument. periods_back is exposed rather than hardcoded since
    a filer that doesn't tag its fourth quarter separately has only three
    periods a year, and offset=4 would then reach back roughly 16 months
    instead of 12.

    Expressed as a plain percent change of the prior period's own value,
    unlike profit_growth_factor: total_assets is virtually always
    positive, so it doesn't have the near-zero-denominator problem
    net_income does.

    Returns None if either total assets figure cannot be resolved, or if
    the prior period's total assets is non-positive.
    """
    current = latest_value_as_of(facts, "total_assets", "USD", as_of, offset=0)
    prior = latest_value_as_of(facts, "total_assets", "USD", as_of, offset=periods_back)
    if current is None or prior is None or prior[0] <= 0:
        return None
    return (current[0] - prior[0]) / prior[0]


In [30]:
toy_investment_facts = {
    "facts": {
        "us-gaap": {
            "Assets": {
                "units": {"USD": [
                    {"end": "2023-03-31", "val": 900.0, "filed": "2023-05-01", "form": "10-Q"},
                    {"end": "2023-06-30", "val": 950.0, "filed": "2023-08-01", "form": "10-Q"},
                    {"end": "2023-09-30", "val": 980.0, "filed": "2023-11-01", "form": "10-Q"},
                    {"end": "2023-12-31", "val": 1000.0, "filed": "2024-02-01", "form": "10-K"},
                    {"end": "2024-03-31", "val": 1100.0, "filed": "2024-05-01", "form": "10-Q"},
                ]}
            },
        }
    }
}
print(investment_factor(toy_investment_facts, "2024-06-28"))   # (1100 - 900) / 900 ≈ 0.2222


0.2222222222222222


In [31]:
rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    if facts is None:
        continue
    ticker = ticker_on(ticker_history, cik, AS_OF)
    inv = investment_factor(facts, AS_OF)
    if inv is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "investment": inv})

investment_df = pd.DataFrame(rows)
print(f"{len(investment_df)} / {len(sample_ciks)} resolved")
print(investment_df["investment"].describe())
print(investment_df.sort_values("investment").head(5)[["ticker", "investment"]])
print(investment_df.sort_values("investment").tail(5)[["ticker", "investment"]])


60 / 60 resolved
count    60.000000
mean      0.131700
std       0.288396
min      -0.412702
25%       0.010649
50%       0.058815
75%       0.133361
max       1.444793
Name: investment, dtype: float64
   ticker  investment
2     FIS   -0.412702
27   SCHW   -0.124671
56    AFL   -0.075752
40    ADM   -0.067532
29    KEY   -0.050800
   ticker  investment
19    NEM    0.441992
25   NDAQ    0.456895
43    TPR    0.965509
58    EXR    1.264969
20   AVGO    1.444793


### 3n. Accruals: net income minus operating cash flow, over total assets


In [32]:
def accruals_factor(facts, as_of):
    """Raw accruals factor (Sloan 1996): net income minus operating cash
    flow, scaled by total assets.

    Measures the non-cash component of reported earnings: a high value
    means net income is running well ahead of the cash a company actually
    generated, the classic red flag this factor exists to capture (high
    accruals firms have been shown to subsequently underperform, on
    average, since the non-cash portion of earnings tends not to persist).

    Scaled by total assets, the same convention as profit_growth_factor
    and gross_profitability_factor, for the same reason: a well behaved,
    virtually always positive denominator.

    Returns None if net income, operating cash flow, or total assets
    cannot be resolved, or if total assets is non-positive.
    """
    ni = latest_value_as_of(facts, "net_income", "USD", as_of, period="annual")
    ocf = latest_value_as_of(facts, "operating_cash_flow", "USD", as_of, period="annual")
    if ni is None or ocf is None:
        return None
    assets = latest_value_as_of(facts, "total_assets", "USD", as_of)
    if assets is None or assets[0] <= 0:
        return None
    return (ni[0] - ocf[0]) / assets[0]


In [33]:
toy_accruals_facts = {
    "facts": {
        "us-gaap": {
            "NetIncomeLoss": {
                "units": {"USD": [
                    {"start": "2023-01-01", "end": "2023-12-31", "val": 500.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
            "NetCashProvidedByUsedInOperatingActivities": {
                "units": {"USD": [
                    {"start": "2023-01-01", "end": "2023-12-31", "val": 300.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
            "Assets": {
                "units": {"USD": [
                    {"end": "2023-12-31", "val": 1000.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
        }
    }
}
print(accruals_factor(toy_accruals_facts, "2024-06-28"))   # (500 - 300) / 1000 = 0.2


0.2


In [34]:
rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    if facts is None:
        continue
    ticker = ticker_on(ticker_history, cik, AS_OF)
    acc = accruals_factor(facts, AS_OF)
    if acc is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "accruals": acc})

accruals_df = pd.DataFrame(rows)
print(f"{len(accruals_df)} / {len(sample_ciks)} resolved")
print(accruals_df["accruals"].describe())
print(accruals_df.sort_values("accruals").head(5)[["ticker", "accruals"]])
print(accruals_df.sort_values("accruals").tail(5)[["ticker", "accruals"]])


60 / 60 resolved
count    60.000000
mean     -0.032542
std       0.061977
min      -0.306348
25%      -0.050170
50%      -0.031208
75%      -0.010262
max       0.270930
Name: accruals, dtype: float64
   ticker  accruals
2     FIS -0.306348
21   PANW -0.130379
39   POOL -0.097105
19    NEM -0.095003
17     KR -0.089649
   ticker  accruals
51   EBAY  0.015921
48   TSLA  0.015939
44    KDP  0.016295
38    BAX  0.033468
41    EMR  0.270930


### 3o. Low leverage: total debt over stockholders' equity


In [35]:
def leverage_factor(facts, as_of):
    """Raw low leverage factor: total debt (long term, current plus
    noncurrent) over stockholders' equity.

    Named "low leverage" in the README and the JKP taxonomy because low
    values of this ratio are the ones associated with the premium, the
    same convention as low_vol_factor: this function returns the plain
    debt-to-equity ratio itself, not its negative, since which direction
    to bet is a decision for the alpha model later, not baked into a raw
    factor's sign here.

    Missing long_term_debt_current specifically is treated as zero, not
    as missing data: fundamentals.py's own documentation established this
    concept is frequently and legitimately zero and often left untagged
    once it is, not evidence of unresolvable data. Missing
    long_term_debt_noncurrent is treated the same way for symmetry,
    though it's a much rarer case in practice. Only a company with no
    resolvable debt tag at all makes this factor unresolvable.

    Like quality.py's stockholders_equity, a non-positive value there is
    not filtered out: negative book equity is real, common data
    (leveraged buybacks), not an error, left for scoring/zscore.py and
    later IC measurement to handle. Equity of exactly zero is the one
    case still guarded against, since dividing by it would raise rather
    than produce a meaningful ratio.

    Returns None if stockholders' equity cannot be resolved or is
    exactly zero, or if both debt concepts are unresolvable.
    """
    noncurrent = latest_value_as_of(facts, "long_term_debt_noncurrent", "USD", as_of)
    current = latest_value_as_of(facts, "long_term_debt_current", "USD", as_of)
    if noncurrent is None and current is None:
        return None
    total_debt = (noncurrent[0] if noncurrent else 0) + (current[0] if current else 0)

    equity = latest_value_as_of(facts, "stockholders_equity", "USD", as_of)
    if equity is None or equity[0] == 0:
        return None
    return total_debt / equity[0]


In [36]:
toy_leverage_facts = {
    "facts": {
        "us-gaap": {
            "LongTermDebtNoncurrent": {
                "units": {"USD": [
                    {"end": "2023-12-31", "val": 400.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
            "LongTermDebtCurrent": {
                "units": {"USD": [
                    {"end": "2023-12-31", "val": 100.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
            "StockholdersEquity": {
                "units": {"USD": [
                    {"end": "2023-12-31", "val": 500.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
        }
    }
}
print(leverage_factor(toy_leverage_facts, "2024-06-28"))   # (400 + 100) / 500 = 1.0


1.0


In [37]:
rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    if facts is None:
        continue
    ticker = ticker_on(ticker_history, cik, AS_OF)
    lev = leverage_factor(facts, AS_OF)
    if lev is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "leverage": lev})

leverage_df = pd.DataFrame(rows)
print(f"{len(leverage_df)} / {len(sample_ciks)} resolved")
print(leverage_df["leverage"].describe())
print(leverage_df.sort_values("leverage").head(5)[["ticker", "leverage"]])
print(leverage_df.sort_values("leverage").tail(5)[["ticker", "leverage"]])


57 / 60 resolved
count     57.000000
mean      12.952782
std       90.693443
min       -4.896754
25%        0.343740
50%        0.765460
75%        1.428198
max      685.586188
Name: leverage, dtype: float64
   ticker  leverage
11     MO -4.896754
4     DPZ -1.242083
33   FFIV  0.000000
13   SNPS  0.002371
56   PAYC  0.020166
   ticker    leverage
43    LMT    2.920000
34     IT    3.421005
7     COR    4.846510
8    TRGP    4.884056
44    IRM  685.586188


### 3p. Debt issuance (proxy): percent change in total debt, year over year


In [38]:
def debt_issuance_factor(facts, as_of, periods_back=4):
    """Raw debt issuance factor (proxy): percent change in total debt
    (long term, current plus noncurrent) between the most recent balance
    sheet and the one roughly a year earlier.

    A proxy, not the literature's precise definition (Bradshaw, Richardson,
    Sloan 2006): the precise version nets cash-flow-statement proceeds
    from debt issuance against repayments during the period, capturing
    actual financing activity. This measures the balance outstanding
    instead, which can move for reasons unrelated to issuance (FX
    translation on foreign debt, fair value remeasurement, a noncurrent
    tranche reclassified to current, or a lease accounting standard
    change folding new leases into the long term debt tags) and misses
    a same-period issuance-and-repayment refinancing entirely, since
    that leaves the balance unchanged. Built as a proxy deliberately: per
    the README's build order reasoning, whether the imprecision here
    matters enough to justify the loader work a precise version needs is
    a question for the step 6 IC pass, not something to guess at before
    any measurement exists.

    Same offset=4 mechanism as investment_factor, for the same reason:
    long_term_debt_noncurrent and long_term_debt_current are both instant
    concepts reported every quarter, so offset=4 is what reaches roughly
    a year back for a filer reporting all four quarters separately.
    Missing long_term_debt_current specifically (in either period) is
    treated as zero, not as missing data, same reasoning as
    leverage_factor.

    Returns None if total debt is unresolvable (both debt concepts
    missing) for either period, or if the prior period's total debt is
    exactly zero.
    """
    def _total_debt(offset):
        noncurrent = latest_value_as_of(facts, "long_term_debt_noncurrent", "USD", as_of, offset=offset)
        current = latest_value_as_of(facts, "long_term_debt_current", "USD", as_of, offset=offset)
        if noncurrent is None and current is None:
            return None
        return (noncurrent[0] if noncurrent else 0) + (current[0] if current else 0)

    current_debt = _total_debt(0)
    prior_debt = _total_debt(periods_back)
    if current_debt is None or prior_debt is None or prior_debt == 0:
        return None
    return (current_debt - prior_debt) / prior_debt


In [39]:
toy_debt_issuance_facts = {
    "facts": {
        "us-gaap": {
            "LongTermDebtNoncurrent": {
                "units": {"USD": [
                    {"end": "2023-03-31", "val": 800.0, "filed": "2023-05-01", "form": "10-Q"},
                    {"end": "2023-06-30", "val": 820.0, "filed": "2023-08-01", "form": "10-Q"},
                    {"end": "2023-09-30", "val": 850.0, "filed": "2023-11-01", "form": "10-Q"},
                    {"end": "2023-12-31", "val": 900.0, "filed": "2024-02-01", "form": "10-K"},
                    {"end": "2024-03-31", "val": 1000.0, "filed": "2024-05-01", "form": "10-Q"},
                ]}
            },
        }
    }
}
print(debt_issuance_factor(toy_debt_issuance_facts, "2024-06-28"))   # (1000 - 800) / 800 = 0.25


0.25


In [40]:
rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    if facts is None:
        continue
    ticker = ticker_on(ticker_history, cik, AS_OF)
    di = debt_issuance_factor(facts, AS_OF)
    if di is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "debt_issuance": di})

debt_issuance_df = pd.DataFrame(rows)
print(f"{len(debt_issuance_df)} / {len(sample_ciks)} resolved")
print(debt_issuance_df["debt_issuance"].describe())
print(debt_issuance_df.sort_values("debt_issuance").head(5)[["ticker", "debt_issuance"]])
print(debt_issuance_df.sort_values("debt_issuance").tail(5)[["ticker", "debt_issuance"]])


54 / 60 resolved
count    54.000000
mean      0.232214
std       0.828992
min      -1.000000
25%      -0.090657
50%       0.000445
75%       0.142457
max       3.637910
Name: debt_issuance, dtype: float64
   ticker  debt_issuance
33   FFIV      -1.000000
13   SNPS      -0.551501
20   PANW      -0.458361
2     FIS      -0.302294
37   POOL      -0.290041
   ticker  debt_issuance
24   NDAQ       0.955177
45   TSLA       1.781963
28   CPAY       2.353158
6    ORCL       3.472796
41    TPR       3.637910


### Panel check: known hard cases

`notebooks/panel.py`'s 28-company panel (shared with `validating_fundamentals.ipynb`) exists
specifically because a random sample can go a long time without ever hitting a genuinely hard
case: a dual class share count (`GOOGL`, `META`), a tag naming era transition (`CSX`, `HD`,
`TSN`), or a filer that returns `None` for everything (`CCU-200807`, a foreign private issuer).
Running every factor against it is a different, complementary check from the random-sample
`describe()` calls above: not "is the distribution sane" but "does a known-tricky company
resolve to `None` where it should, and a real number where it shouldn't, without crashing."

In [ ]:
from notebooks.panel import resolve_panel_ciks

panel_rows = []
for ticker, cik, axis, why in resolve_panel_ciks(ticker_history):
    facts = load_company_facts(cik)
    p_ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    have_facts = facts is not None
    have_price_inputs = prices is not None and p_ticker is not None
    market_cap = market_cap_as_of(facts, prices, p_ticker, AS_OF) if have_facts and have_price_inputs else None

    panel_rows.append({
        "ticker": ticker,
        "axis": axis,
        "size": size_factor(market_cap),
        "value": earnings_yield_factor(facts, prices, p_ticker, AS_OF) if have_facts and have_price_inputs else None,
        "quality": roe_factor(facts, AS_OF) if have_facts else None,
        "momentum": momentum_factor(prices, p_ticker, AS_OF) if have_price_inputs else None,
        "low_vol": low_vol_factor(prices, p_ticker, AS_OF) if have_price_inputs else None,
        "short_term_reversal": short_term_reversal_factor(prices, p_ticker, AS_OF) if have_price_inputs else None,
        "seasonality": seasonality_factor(prices, p_ticker, AS_OF) if have_price_inputs else None,
        "high_proximity": high_proximity_factor(prices, p_ticker, AS_OF) if have_price_inputs else None,
        "volume_shock": volume_shock_factor(prices, p_ticker, AS_OF) if have_price_inputs else None,
        "illiquidity": illiquidity_factor(prices, p_ticker, AS_OF) if have_price_inputs else None,
        "profitability": gross_profitability_factor(facts, AS_OF) if have_facts else None,
        "profit_growth": profit_growth_factor(facts, AS_OF) if have_facts else None,
        "investment": investment_factor(facts, AS_OF) if have_facts else None,
        "accruals": accruals_factor(facts, AS_OF) if have_facts else None,
        "leverage": leverage_factor(facts, AS_OF) if have_facts else None,
        "debt_issuance": debt_issuance_factor(facts, AS_OF) if have_facts else None,
    })

panel_df = pd.DataFrame(panel_rows).set_index("ticker")
panel_df

### Summary: all factors built so far

One combined table and one chart, both meant to be extended, not rebuilt, as new factors are
added: append a new key to `factor_series` below and both the table and the chart pick it up
automatically.

In [ ]:
factor_series = {
    "size": size_df.set_index("cik")["size_factor"],
    "value": value_df.set_index("cik")["earnings_yield"],
    "quality": quality_df.set_index("cik")["roe"],
    "momentum": momentum_df.set_index("cik")["momentum"],
    "low_vol": lowvol_df.set_index("cik")["low_vol"],
    "short_term_reversal": reversal_df.set_index("cik")["reversal"],
    "seasonality": seasonality_df.set_index("cik")["seasonality"],
    "high_proximity": high_df.set_index("cik")["high_proximity"],
    "volume_shock": volume_shock_df.set_index("cik")["volume_shock"],
    "illiquidity": illiquidity_df.set_index("cik")["illiquidity"],
    "profitability": profitability_df.set_index("cik")["gross_profitability"],
    "profit_growth": profit_growth_df.set_index("cik")["profit_growth"],
    "investment": investment_df.set_index("cik")["investment"],
    "accruals": accruals_df.set_index("cik")["accruals"],
    "leverage": leverage_df.set_index("cik")["leverage"],
    "debt_issuance": debt_issuance_df.set_index("cik")["debt_issuance"],
}

summary = pd.DataFrame({name: s.describe() for name, s in factor_series.items()})
summary

In [ ]:
import matplotlib.pyplot as plt

ncols = 4
nrows = math.ceil(len(factor_series) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows))
axes = axes.flatten()
for ax, (name, series) in zip(axes, factor_series.items()):
    ax.hist(series.dropna(), bins=20, color="#3987e5", edgecolor="white", linewidth=0.5)
    ax.set_title(name)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
for ax in axes[len(factor_series):]:
    ax.set_visible(False)

fig.suptitle(f"Factor distributions, {len(sample_ciks)}-company sample, {AS_OF}")
fig.tight_layout()
plt.show()

## Part 4: scoring — zscore, combine, neutralize

### 4a. Cross sectional z-score, with winsorization

Whether z-scoring needs outlier handling was checked empirically rather than assumed: 3b's
real earnings-yield sample shows no outlier disconnected from its neighbors once the
split-adjustment bug was fixed, so winsorizing at the 1st/99th percentile is cheap insurance
for a factor or date not covered by that specific check (3c's ROE sample has a much wider
spread), not a correction this data was shown to need on its own.

In [44]:
toy = pd.Series(range(1, 101), dtype=float)               # 1..100, evenly spread
toy_with_outlier = pd.concat([toy, pd.Series([10_000.0])], ignore_index=True)
clipped = winsorize(toy_with_outlier, 0.01, 0.99)
print("raw max:", toy_with_outlier.max(), " clipped max:", clipped.max())

toy_with_gap = pd.Series([1.0, 2.0, 3.0, None, 5.0])
z = zscore(toy_with_gap, winsorize_pct=0)
print(z)
print("NaN count:", z.isna().sum())

raw max: 10000.0  clipped max: 100.0
0   -1.024695
1   -0.439155
2    0.146385
3         NaN
4    1.317465
dtype: float64
NaN count: 1


### 4b. Combine: weighted average with missing-data handling

A stock missing one or more factors has those terms dropped and the remaining weights
renormalized, per the README's missing-data rule, never a substituted zero.

In [45]:
factors = pd.DataFrame({
    "momentum": [1.0, 1.0],
    "value": [-1.0, None],
}, index=["A", "B"])
weights = {"momentum": 0.5, "value": 0.5}
print(combine(factors, weights))   # expect A: 0.0, B: 1.0, not 0.5

factors_all_missing = pd.DataFrame({
    "momentum": [1.0, None],
    "value": [-1.0, None],
}, index=["A", "B"])
print(combine(factors_all_missing, weights))   # expect A: 0.0, B: NaN, not an error

A    0.0
B    1.0
dtype: float64
A    0.0
B    NaN
dtype: float64


### 4c. Neutralize: residualize the combined score against beta

Refit fresh at every rebalance date, never reused, since the regression coefficients are
expected to change from one date to the next.

In [46]:
betas = pd.Series([1.0, 2.0, 3.0, 4.0], index=["A", "B", "C", "D"])
scores = pd.Series([1.0, 2.0, 6.0, None], index=["A", "B", "C", "D"])
# A, B, C fit a line with slope 2.5, intercept -2: predicted 0.5, 3.0, 5.5,
# residuals 0.5, -1.0, 0.5. D has a beta but no score, excluded from the fit.
print(neutralize(scores, betas))   # expect A: 0.5, B: -1.0, C: 0.5, D: NaN

A    0.5
B   -1.0
C    0.5
D    NaN
dtype: float64
